In [2]:
import pandas as pd

file_path = 'datasets/titanic.csv'
df = pd.read_csv(file_path)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.shape
# df.dtypes
# df.isnull().sum() 
# df.describe()     
# df.head(10)
# df.tail(5)

(891, 12)

Average Age: 29.6

Most Nulls in: Cabin

Rows: 891, Columns: 12

### Task #1

 1. Filter passengers older than 30
 2. Select only Name, Age, and Survived columns
 3. Filter female passengers who survived
 4. Filter passengers where Age is null
 5. Filter 1st class passengers who did NOT survive

In [4]:
# 1
# df[df.Age>30]

# 2
# df[['Name', 'Age', 'Survived']]

# 3
# df[(df['Sex'] =='female') & (df['Survived'] == 1)]

# 4
# df[df['Age'].isnull()]

# 5
# df[(df['Pclass'] == 1) & (df['Survived']==0)]

### Task 2
 
 1. Average fare by passenger class

 2. Survival count by sex

 3. Average age by survival status (survived vs not)

In [ ]:
# df.groupby('Pclass').Fare.mean()

# df.groupby('Sex').Survived.count()

# df.groupby('Survived').Age.mean()

Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64

### SQL Joins

Left Joins and Right Joins can be done with merge method under df class

In [10]:
# Create small DataFrames and join them
class_info = pd.DataFrame({
    'Pclass': [1, 2, 3],
    'Class_Label': ['First', 'Second', 'Third']
})

# LEFT JOIN equivalent
merged = df.merge(class_info, on='Pclass', how='inner')
merged[['Name', 'Pclass', 'Class_Label']].head()

,Name,Pclass,Class_Label
0,"Braund, Mr. Owen Harris",3,Third
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,First
2,"Heikkinen, Miss. Laina",3,Third
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,First
4,"Allen, Mr. William Henry",3,Third


### Applying Custom Functions

apply() method lets you use custom functions

In [ ]:
def age_group(age):
    if pd.isnull(age):
        return 'Unknown'
    elif age < 18:
        return 'Child'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

df['AgeGroup'] = df['Age'].apply(age_group)

# Now group by your new column
df.groupby('AgeGroup')['Survived'].mean()*100

AgeGroup
Adult      38.608696
Child      53.982301
Senior     26.923077
Unknown    29.378531
Name: Survived, dtype: float64

### SQL
```sql
SELECT Pclass,
       COUNT(*) AS total_passengers,
       SUM(Survived) AS survived,
       ROUND(AVG(Survived) * 100, 2) AS survival_rate_pct
FROM titanic
GROUP BY Pclass
ORDER BY Pclass;
```

In [16]:
df.groupby('Pclass').agg(total_survived=('Survived', 'sum'), average_age=('Age', 'mean'), survival_rate_pct=('Survived', lambda x: round(x.mean() * 100, 2))).reset_index()

,Pclass,total_survived,average_age,survival_rate_pct
0,1,136,38.233441,62.96
1,2,87,29.877630,47.28
2,3,119,25.140620,24.24


-- SQL
```sql
SELECT Name, Fare, Pclass
FROM titanic
ORDER BY Fare DESC
LIMIT 5;
```

In [18]:
df[['Name', 'Fare', 'Pclass']].sort_values(by='Fare', ascending=False).head(5)

,Name,Fare,Pclass
679,"Cardeza, Mr. Thomas Drake Martinez",512.3292,1
258,"Ward, Miss. Anna",512.3292,1
737,"Lesurer, Mr. Gustave J",512.3292,1
88,"Fortune, Miss. Mabel Helen",263.0000,1
438,"Fortune, Mr. Mark",263.0000,1


-- SQL
```sql
SELECT Name, Age,
       CASE WHEN Age IS NULL THEN 'Missing'
            WHEN Age < 18 THEN 'Child'
            ELSE 'Adult'
       END AS age_category
FROM titanic;

```

In [27]:
def apply_age_group(age):
    if pd.isnull(age):
        return 'Unknown'
    elif age < 18:
        return 'Child'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

df['age_category'] = df['Age'].apply(apply_age_group)
df[['Name', 'Age', 'age_category']].head()

,Name,Age,age_category
0,"Braund, Mr. Owen Harris",22.0,Adult
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,Adult
2,"Heikkinen, Miss. Laina",26.0,Adult
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,Adult
4,"Allen, Mr. William Henry",35.0,Adult


-- SQL
```sql
SELECT Name, Sex, Age, Fare
FROM titanic
WHERE Survived = 1
  AND Fare > 50
  AND Age IS NOT NULL
ORDER BY Fare DESC;
```

In [31]:
df[(df['Survived']==1) & (df['Fare']>50) & (df['Age'].notnull())][['Name', 'Sex', 'Age', 'Fare']].sort_values(by='Fare', ascending=False)

,Name,Sex,Age,Fare
258,"Ward, Miss. Anna",female,35.0,512.3292
737,"Lesurer, Mr. Gustave J",male,35.0,512.3292
679,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,512.3292
88,"Fortune, Miss. Mabel Helen",female,23.0,263.0000
341,"Fortune, Miss. Alice Elizabeth",female,24.0,263.0000
...,...,...,...,...
621,"Kimball, Mr. Edwin Nelson Jr",male,42.0,52.5542
871,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,52.5542
383,"Holverson, Mrs. Alexander Oskar (Mary Aline To...",female,35.0,52.0000
712,"Taylor, Mr. Elmer Zebley",male,48.0,52.0000


-- SQL
```sql
SELECT Embarked,
       COUNT(*) AS total,
       ROUND(AVG(Fare), 2) AS avg_fare
FROM titanic
WHERE Embarked IS NOT NULL
GROUP BY Embarked
HAVING COUNT(*) > 20
ORDER BY avg_fare DESC;
```

In [38]:
df[df['Embarked'].notnull()].groupby('Embarked').agg(total=('Fare', 'count'), avg_fare=('Fare', lambda x: round(x.mean(), 2))).query('total>20').sort_values(by='avg_fare', ascending=False)

,total,avg_fare
Embarked,,
C,168,59.95
S,644,27.08
Q,77,13.28
